In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# ---- 1. Data Preparation with Uniform Dequantization ----
def uniform_dequantize(x):
    u = torch.rand_like(x)
    x_deq = (x * 255.0 + u) / 256.0  # x in [0,1]; make [0,255] + U[0,1] then /256
    return x_deq

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(uniform_dequantize)
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

# ---- 2. Affine Coupling Layer ----
class AffineCoupling(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.dim = dim
        self.net = nn.Sequential(
            nn.Linear(dim//2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        xA, xB = x.chunk(2, dim=1)
        st = self.net(xA)
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s)  # Optional: constrain scaling
        yA = xA
        yB = xB * torch.exp(s) + t
        y = torch.cat([yA, yB], dim=1)
        log_det = s.sum(dim=1)
        return y, log_det

    def inverse(self, y):
        yA, yB = y.chunk(2, dim=1)
        st = self.net(yA)
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s)
        xA = yA
        xB = (yB - t) * torch.exp(-s)
        x = torch.cat([xA, xB], dim=1)
        return x

# ---- 3. Permutation Layer ----
class Flip(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.perm = torch.arange(dim-1, -1, -1)

    def forward(self, x):
        return x[:, self.perm], 0  # 0 log-det

    def inverse(self, y):
        return y[:, self.perm]

# ---- 4. Full Flow Model ----
class SimpleFlow(nn.Module):
    def __init__(self, n_blocks, dim, hidden_dim):
        super().__init__()
        layers = []
        for i in range(n_blocks):
            layers.append(AffineCoupling(dim, hidden_dim))
            layers.append(Flip(dim))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        log_det_sum = 0
        for layer in self.layers:
            if isinstance(layer, AffineCoupling):
                x, log_det = layer(x)
                log_det_sum += log_det
            else:
                x, _ = layer(x)
        return x, log_det_sum

    def inverse(self, z):
        for layer in reversed(self.layers):
            if isinstance(layer, AffineCoupling):
                z = layer.inverse(z)
            else:
                z = layer.inverse(z)
        return z

# ---- 5. Training Loop ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dim = 28*28
hidden_dim = 512
n_blocks = 8
flow = SimpleFlow(n_blocks, dim, hidden_dim).to(device)
optimizer = optim.Adam(flow.parameters(), lr=1e-3)
epochs = 20

for epoch in range(epochs):
    total_loss = 0
    for x, _ in train_loader:
        x = x.view(-1, dim).to(device)
        z, log_det = flow(x)
        # Log-probability under standard normal
        log_prob = -0.5 * (z ** 2).sum(dim=1) - 0.5 * dim * np.log(2 * np.pi)
        # Negative log-likelihood
        loss = -(log_prob + log_det).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# ---- 6. Sampling and Visualization ----
flow.eval()
with torch.no_grad():
    z = torch.randn(64, dim).to(device)
    samples = flow.inverse(z).cpu()
    samples = samples.clamp(0, 1).view(-1, 1, 28, 28)

# Plot samples
grid = samples[:64].reshape(8, 8, 28, 28)
fig, axes = plt.subplots(8, 8, figsize=(8,8))
for i in range(8):
    for j in range(8):
        axes[i, j].imshow(grid[i, j], cmap='gray')
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()


Epoch 1/20, Loss: -1261.9984
Epoch 2/20, Loss: -1622.0716


KeyboardInterrupt: 